In [1]:
from pathlib import Path

from rdflib import Graph
from rdfine import GraphReader
from compilers import PipelineGenerator, ProjectBuilder

In [2]:
data_dir = Path("../data")

graph = Graph()
for filename in [
    "catalog.ttl",
    "pipeline_definition_nifi.ttl",
    "tcs_shapes.ttl",
]:
    graph.parse(
        data_dir / filename,
        publicID="file:///workspace/pipeline/",
    )

reader = GraphReader(graph).infer(data_dir / "inference_rules.yaml")

In [3]:
report = reader.validate(advanced=True, inference="rdfs")

violations = report.select(
    "?focus ?message",
    """
    ?result a sh:ValidationResult ;
        sh:focusNode ?focus ;
        sh:resultMessage ?message .
    """,
)

for row in violations.itertuples(index=False):
    print(f"Focus:   {row.focus}")
    print(f"Message: {row.message}")
    print()
    
if not report.ask("?report sh:conforms true"):
    for row in violations.itertuples(index=False):
        print(f"{row.focus}: {row.message}")

    raise ValueError("Pipeline definition does not conform")

In [4]:
generator = PipelineGenerator(":DemonstratorPipeline", reader.graph)
build_graph = generator.compile()

[compiler.__name__ for compiler in generator.compilers]

['PipelineExtractor',
 'PipelineAssembler',
 'NifiConfigCompiler',
 'DockerComposeCompiler']

In [5]:
builder = ProjectBuilder(build_graph)

for _, file in builder.files.iterrows():
    print(f"=== {file['filepath']}/{file['filename']} ===")
    print(file["content"])

=== nifi/flow.json ===
{
    "maxTimerDrivenThreadCount": 10,
    "rootGroup": {
        "identifier": "92b7e0fa-f01d-50ba-a3f9-d4b952e8a26b",
        "instanceIdentifier": "c36b4aa0-6ae8-5922-99d3-7cf17834cd86",
        "name": "Demonstrator Pipeline.",
        "comments": "Polls API, transforms to RDF, detects threshold, triggers email alert.",
        "position": {
            "x": 0.0,
            "y": 0.0
        },
        "processGroups": [],
        "remoteProcessGroups": [],
        "processors": [
            {
                "identifier": "16af64dd-2fc6-51c5-bb8e-8c53fd1ab340",
                "instanceIdentifier": "0648ccaa-ee95-55de-9a00-f11148ac47fe",
                "name": "GenerateFlowFile",
                "comments": "",
                "position": {
                    "x": 0.0,
                    "y": 0.0
                },
                "type": "org.apache.nifi.processors.standard.GenerateFlowFile",
                "bundle": {
                    "group": "org

In [6]:
written = builder.write("../out/nifi_testing")

for path in written:
    print(path)

C:\Users\ThomasDelaeter\Documents\Projecten\toolchain-specification\pipeline generator\out\nifi_testing\nifi\flow.json
C:\Users\ThomasDelaeter\Documents\Projecten\toolchain-specification\pipeline generator\out\nifi_testing\docker-compose.yml
